# 🚀 Módulo 01 — Ejecución de Agentes

> **Objetivo:** Dominar las dos formas de ejecutar un agente y cómo personalizar parámetros por invocación.

## ¿Qué vas a aprender?

- `await agent.run(prompt)` → respuesta completa de una sola vez
- `async for update in agent.run(prompt, stream=True)` → respuesta en streaming (token por token)
- `options={...}` → ajustar `temperature`, `max_tokens`, etc. para una llamada específica

## ¿Cuándo usar cada modo?

| Modo | Cuándo |
|------|--------|
| **Completo** (`run`) | Necesitas la respuesta entera para procesarla (parsing, almacenamiento, validación) |
| **Streaming** (`run(..., stream=True)`) | UX en tiempo real (chat estilo ChatGPT, transcripción de voz) |


## ⚙️ Setup inicial

Esta celda carga la configuración de Azure OpenAI desde `appsettings.Development.json`
(o variables de entorno) y crea el cliente que reutilizaremos durante todo el notebook.

> 💡 Solo necesitas ejecutar esta celda **una vez** al abrir el notebook.


In [ ]:
# Aseguramos que la raíz del proyecto esté en sys.path para importar `helpers`
import sys
from pathlib import Path

_ROOT = Path.cwd()
# Si abriste el notebook desde la carpeta notebooks/, sube un nivel
if _ROOT.name == "notebooks":
    _ROOT = _ROOT.parent
if str(_ROOT) not in sys.path:
    sys.path.insert(0, str(_ROOT))

from agent_framework import Agent
from helpers.config import create_chat_client
print("✅ Helpers cargados. Endpoint:", create_chat_client().__class__.__name__)


## 1️⃣ Respuesta completa con `await agent.run(...)`

Espera a que se genere TODO el texto antes de devolverlo. Más simple, ideal para procesamiento batch.


In [ ]:
agent = Agent(
    client=create_chat_client(),
    instructions="Eres un experto en geografía. Siempre responde en una oración.",
)

response = await agent.run("What is the capital of France?")
print("✅ Respuesta completa:")
print(f"   {response.text}")


## 2️⃣ Streaming token por token

`stream=True` convierte la llamada en un async iterator. Cada `update.text` contiene un fragmento incremental — perfecto para mostrar la respuesta a medida que se genera.


In [ ]:
agent = Agent(
    client=create_chat_client(),
    instructions="Eres un profesor de ciencias. Explica los conceptos de forma simple en 2-3 oraciones.",
)

print("✅ Streaming (mira cómo aparece el texto poco a poco):\n")
async for update in agent.run("Explain photosynthesis.", stream=True):
    if update.text:
        print(update.text, end="", flush=True)
print()


## 3️⃣ Opciones personalizadas por invocación

Puedes pasar `options={...}` para ajustar parámetros sólo en esa llamada (sin tocar el agente).

Parámetros comunes:
- `temperature` — creatividad (0.0 = determinista, 1.0+ = creativo)
- `max_tokens` — longitud máxima de la respuesta
- `top_p`, `frequency_penalty`, `presence_penalty`, etc.


In [ ]:
agent = Agent(
    client=create_chat_client(),
    instructions="Eres un escritor creativo. Escribe poemas cortos.",
)

response = await agent.run(
    "Write a haiku about programming.",
    options={
        "temperature": 0.9,   # alta creatividad
        "max_tokens": 100,    # corto
    },
)
print(f"✅ Haiku creativo:\n   {response.text}")


## 🎯 Resumen

| Modo | Sintaxis | Cuándo |
|------|----------|--------|
| Completo | `await agent.run("...")` | Procesamiento, validación, batch |
| Streaming | `async for u in agent.run("...", stream=True):` | UX interactiva |
| Opciones | `await agent.run("...", options={"temperature": 0.9})` | Ajuste por invocación |

**Siguiente módulo →** [Módulo 02 — Salida Estructurada (Pydantic)](./02_structured_output.ipynb)
